# Lab 3 — Frameworks : **LangGraph**, **RAG** & **CrewAI**
**Séance 3 · Frameworks industriels & systèmes multi-agents · PGE5 / M2**

Au Lab 2, vous avez codé la boucle d'agent **à la main**. On passe maintenant aux outils de
**production**, en gardant le réflexe « *que masque le framework, exactement ?* ».

### Plan
- **A. LangGraph** — l'agent comme **graphe d'états** : du *prebuilt* au graphe **construit à la main**,
  puis **persistance** + ***human-in-the-loop***.
- **B. RAG** — donner une **mémoire documentaire** : retriever → **RAG agentique** → contrôle de *groundedness*.
- **C. CrewAI** — un **système multi-agents** (chercheur → rédacteur → relecteur), séquentiel puis
  hiérarchique, et une version **« à la main »** pour démystifier.

> ⚠️ **Dépendances.** Les parties LangGraph/CrewAI nécessitent les paquets et **une clé d'API** :
> ```bash
> pip install langgraph langchain crewai langchain-google-genai  # adaptateur selon LLM_PROVIDER
> ```
> Les cellules « frameworks » se **désactivent proprement** si un paquet manque (message d'aide).
> Le **RAG (B)** et le **superviseur à la main (C.3)** tournent **hors-ligne, sans dépendance**.

## 0. Sélection du modèle (agnostique, pour les frameworks)

Au Lab 1–2, `llm_helpers` gérait l'agnosticité. Côté **LangChain**, chaque fournisseur a sa
classe (`ChatOpenAI`, `ChatAnthropic`, `ChatMistralAI`) ; côté **CrewAI** (litellm), on passe une
chaîne `"provider/model"`. On centralise les deux ici — **c'est la correction du défaut le plus
courant** (coder en dur `ChatOpenAI` casse l'agnosticité).

In [1]:
import os
# Charge .env puis .secrets pour que LLM_PROVIDER et les cles soient
# disponibles DES LE DEBUT (les cellules LangGraph ci-dessous en dependent).
try:
    from dotenv import load_dotenv
    load_dotenv(".env", override=True); load_dotenv(".secrets", override=True)
except Exception:
    pass

def make_lc_chat(temperature: float = 0.0):
    "Renvoie le bon chat-model LangChain selon LLM_PROVIDER."
    p = os.getenv("LLM_PROVIDER", "openai").lower()
    model = os.getenv("LLM_MODEL")
    if p == "mock":
        raise RuntimeError("LLM_PROVIDER=mock : les cellules LangGraph/CrewAI avec framework necessitent une API. Utilisez B.1, B.3 et C.3 en off.")
    if p == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model or "gemini-2.5-flash", temperature=temperature)
    if p == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model or "claude-sonnet-4-6", temperature=temperature)
    if p == "mistral":
        from langchain_mistralai import ChatMistralAI
        return ChatMistralAI(model=model or "mistral-large-latest", temperature=temperature)
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(model=model or "gpt-4o-mini", temperature=temperature)

def crewai_model_string():
    "Identifiant 'provider/model' pour CrewAI/litellm."
    p = os.getenv("LLM_PROVIDER", "openai").lower()
    model = os.getenv("LLM_MODEL")
    if p == "mock":
        return "mock/offline"
    if p == "openai":
        return model or "gpt-4o-mini"
    defaults = {"mistral": "mistral/mistral-large-latest",
                "anthropic": "anthropic/claude-sonnet-4-6",
                "google": "gemini/gemini-2.5-flash"}
    return f"{p}/{model}" if model else defaults.get(p, "gpt-4o-mini")

def lc_text(msg):
    "Extrait le texte d'un message LangChain (certains modeles renvoient des blocs)."
    c = getattr(msg, "content", msg)
    if isinstance(c, list):
        return " ".join(b.get("text", "") for b in c if isinstance(b, dict)).strip()
    return c

KEY_BY_PROVIDER = {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "google": "GOOGLE_API_KEY",
}
PROVIDER = os.getenv("LLM_PROVIDER", "openai").lower()
HAS_KEY = PROVIDER != "mock" and bool(os.getenv(KEY_BY_PROVIDER.get(PROVIDER, "")))
print("Fournisseur actif :", PROVIDER, "| cle API active detectee :", HAS_KEY,
      "| modele CrewAI :", crewai_model_string())

Fournisseur actif : anthropic | cle API active detectee : True | modele CrewAI : anthropic/claude-sonnet-4-6


## A. LangGraph — l'agent comme **graphe d'états**

LangGraph modélise l'agent par un **graphe** : des **nœuds** (appel du modèle, exécution
d'outils) et des **arêtes conditionnelles** (continuer à appeler des outils, ou s'arrêter).
C'est le pattern de production le plus répandu pour des agents **contrôlables**.

### A.1 — Version *prebuilt* (le raccourci)
`create_agent` recrée, en une ligne, la boucle du Lab 2.

In [2]:
def demo_langgraph_prebuilt():
    from langchain_core.tools import tool
    from langchain.agents import create_agent

    @tool
    def multiply(a: float, b: float) -> float:
        "Multiplie deux nombres."
        return a * b

    @tool
    def word_count(text: str) -> int:
        "Compte le nombre de mots d'un texte."
        return len(text.split())

    graph = create_agent(make_lc_chat(), tools=[multiply, word_count])
    out = graph.invoke({"messages": [("user",
        "Combien de mots dans 'agentic AI for PGE5' ? Multiplie ce nombre par 7.")]})
    print(lc_text(out["messages"][-1]))

try:
    demo_langgraph_prebuilt()
except ImportError as e:
    print("⏭️  LangGraph non installé :", e, "\n   pip install langgraph langchain-openai")
except Exception as e:
    print("⚠️  Besoin d'une clé d'API valide :", type(e).__name__, e)

Voici le résumé :

- 📝 Le texte *'agentic AI for PGE5'* contient **4 mots**.
- ✖️ 4 × 7 = **28** !


### A.2 — Le graphe **construit à la main** (le vrai apprentissage)

`create_agent` cache tout. On reconstruit le **même** agent explicitement pour voir les
**3 ingrédients** d'un graphe LangGraph :

1. un **état** (`MessagesState` : une liste de messages qui s'accumule) ;
2. des **nœuds** : `agent` (appelle le modèle) et `tools` (exécute les outils via `ToolNode`) ;
3. une **arête conditionnelle** `should_continue` : s'il reste des *tool calls*, aller à `tools`,
   sinon **terminer**.

On visualise même le graphe en **Mermaid**.

In [3]:
def build_custom_graph():
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage
    from langgraph.graph import StateGraph, START, END, MessagesState
    from langgraph.prebuilt import ToolNode

    @tool
    def multiply(a: float, b: float) -> float:
        "Multiplie deux nombres."
        return a * b

    tools = [multiply]
    model = make_lc_chat().bind_tools(tools)

    def agent_node(state: MessagesState):
        return {"messages": [model.invoke(state["messages"])]}

    def should_continue(state: MessagesState):
        last = state["messages"][-1]
        return "tools" if getattr(last, "tool_calls", None) else END

    builder = StateGraph(MessagesState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")     # boucle : après les outils, on revient au modèle
    return builder.compile()

try:
    graph = build_custom_graph()
    print(graph.get_graph().draw_mermaid())          # le schéma du graphe
    out = graph.invoke({"messages": [("user", "Combien font 23 fois 19 ?")]})
    print("\nRéponse :", lc_text(out["messages"][-1]))
except ImportError as e:
    print("⏭️  LangGraph non installé :", e)
except Exception as e:
    print("⚠️  Besoin d'une clé d'API valide :", type(e).__name__, e)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


Réponse : **23 fois 19 = 437** 🎯


**À discuter.** Comparez avec votre `react_loop` du Lab 2 : `agent_node` = l'appel modèle,
`ToolNode` = votre exécution d'outils, `should_continue` = votre `if reply.has_tool_calls`.
LangGraph ajoute par-dessus : **état typé**, **streaming**, **persistance**, **reprise** — ce
qu'on exploite tout de suite.

### A.3 — Persistance (mémoire) + ***human-in-the-loop***

Deux capacités de production quasi gratuites avec LangGraph :
- un **checkpointer** (`MemorySaver`) + un `thread_id` → l'agent **se souvient** entre deux appels ;
- `interrupt_before=["tools"]` → l'exécution **s'arrête avant d'agir**, pour une **validation
  humaine** (essentiel pour les actions sensibles : paiement, envoi d'email, suppression…).

In [4]:
def demo_memory_and_hitl():
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, START, END, MessagesState
    from langgraph.prebuilt import ToolNode
    from langgraph.checkpoint.memory import MemorySaver

    @tool
    def send_email(to: str, body: str) -> str:
        "Envoie un email (action SENSIBLE — à valider)."
        return f"Email envoyé à {to}."

    tools = [send_email]
    model = make_lc_chat().bind_tools(tools)
    def agent_node(s): return {"messages": [model.invoke(s["messages"])]}
    def cont(s): return "tools" if getattr(s["messages"][-1], "tool_calls", None) else END

    b = StateGraph(MessagesState)
    b.add_node("agent", agent_node); b.add_node("tools", ToolNode(tools))
    b.add_edge(START, "agent")
    b.add_conditional_edges("agent", cont, {"tools": "tools", END: END})
    b.add_edge("tools", "agent")
    graph = b.compile(checkpointer=MemorySaver(), interrupt_before=["tools"])

    cfg = {"configurable": {"thread_id": "etudiant-1"}}
    graph.invoke({"messages": [("user", "Envoie un email à prof@aivancity.fr disant 'bonjour'.")]}, cfg)
    pend = graph.get_state(cfg).next
    print("⏸️  Interrompu avant :", pend, "→ validation humaine requise.")
    # ... ici un humain valide ...
    out = graph.invoke(None, cfg)     # reprise : exécute l'outil puis conclut
    print("▶️  Après validation :", lc_text(out["messages"][-1]))
    # Mémoire : nouveau message, MÊME thread_id → le contexte est conservé
    out2 = graph.invoke({"messages": [("user", "À qui ai-je écrit ?")]}, cfg)
    print("🧠 Mémoire :", out2["messages"][-1].content)

try:
    demo_memory_and_hitl()
except ImportError as e:
    print("⏭️  LangGraph non installé :", e)
except Exception as e:
    print("⚠️  Besoin d'une clé d'API valide :", type(e).__name__, e)

⏸️  Interrompu avant : ('tools',) → validation humaine requise.
▶️  Après validation : L'email a bien été envoyé à **prof@aivancity.fr** avec le message **"bonjour"**. ✅
🧠 Mémoire : Vous avez écrit à **prof@aivancity.fr**.


## B. RAG — mémoire documentaire

On indexe un petit corpus, on récupère les passages pertinents (***retriever***) et on les
**injecte** dans le prompt. Ici un **TF-IDF pur Python** (zéro dépendance → tourne partout) ;
en production : base vectorielle (Chroma, pgvector, …) avec embeddings.

In [5]:
import math, re
from collections import Counter

def _tok(t):
    return re.findall(r"[a-zàâçéèêëîïôûùüÿœ0-9]+", t.lower())

class TinyTfidf:
    "Retriever TF-IDF minimal (cosinus), sans dépendance externe."
    def __init__(self, docs):
        self.docs = docs
        self.toks = [_tok(d) for d in docs]
        N = len(docs)
        df = Counter(w for ts in self.toks for w in set(ts))
        self.idf = {w: math.log((1 + N) / (1 + df[w])) + 1 for w in df}
        self.vecs = [self._vec(ts) for ts in self.toks]

    def _vec(self, ts):
        tf = Counter(ts)
        n = len(ts) or 1
        return {w: tf[w] / n * self.idf.get(w, 0.0) for w in tf}

    @staticmethod
    def _cos(a, b):
        num = sum(a[w] * b[w] for w in set(a) & set(b))
        na = math.sqrt(sum(v * v for v in a.values()))
        nb = math.sqrt(sum(v * v for v in b.values()))
        return num / (na * nb) if na and nb else 0.0

    def search(self, query, k=2):
        qv = self._vec(_tok(query))
        scored = sorted(((self._cos(qv, v), d) for v, d in zip(self.vecs, self.docs)),
                        key=lambda x: x[0], reverse=True)
        return [(round(s, 3), d) for s, d in scored[:k]]

CORPUS = [
    "Craft AI est une plateforme MLOps/LLMOps souveraine européenne : déploiement, RAG, monitoring.",
    "Le paradigme ReAct combine raisonnement et action via des appels d'outils (Yao et al., 2022).",
    "LangGraph modélise un agent comme un graphe d'états avec des arêtes conditionnelles.",
    "CrewAI orchestre plusieurs agents dotés de rôles, d'objectifs et de tâches.",
    "Un système multi-agents répartit le travail entre un superviseur et des workers spécialisés.",
    "Le RAG (Retrieval-Augmented Generation) injecte des passages récupérés dans le prompt.",
]

retriever = TinyTfidf(CORPUS)
for score, doc in retriever.search("À quoi sert LangGraph ?"):
    print(f"{score:>5}  {doc}")

0.283  LangGraph modélise un agent comme un graphe d'états avec des arêtes conditionnelles.
  0.0  Craft AI est une plateforme MLOps/LLMOps souveraine européenne : déploiement, RAG, monitoring.


### B.1 — Répondre **à partir du contexte** (et seulement de lui)

On récupère les passages, on les met dans le prompt, et on **interdit** au modèle de répondre
hors contexte. (On réutilise la couche agnostique `llm_helpers`, qui tourne aussi en mock.)

In [6]:
from llm_helpers import make_client, credentials_available

def rag_answer(question, k=2, offline_script=None):
    ctx = "\n".join(f"- {d}" for _, d in retriever.search(question, k))
    client = make_client(offline_script=offline_script, quiet=True)
    msg = [
        {"role": "system", "content": "Réponds UNIQUEMENT à partir du CONTEXTE. "
                                      "Si l'info est absente, dis 'Information absente du contexte.'"},
        {"role": "user", "content": f"CONTEXTE:\n{ctx}\n\nQUESTION: {question}"},
    ]
    return client.complete(msg).content

print(rag_answer("Qu'est-ce que Craft AI ?",
                 offline_script=["Craft AI est une plateforme MLOps/LLMOps souveraine européenne."]))
print("---")
print(rag_answer("Quelle est la capitale du Pérou ?",      # hors corpus
                 offline_script=["Information absente du contexte."]))

Craft AI est une plateforme MLOps/LLMOps souveraine européenne qui permet le déploiement, le RAG (Retrieval-Augmented Generation) et le monitoring.
---
Information absente du contexte.


### B.2 — **RAG agentique** : le retriever devient un **outil**

Au lieu de toujours récupérer, on laisse l'**agent décider** quand chercher. On expose la
recherche comme un **outil LangGraph**. L'agent peut chaîner *recherche → raisonnement → réponse*.

In [7]:
def demo_agentic_rag():
    from langchain_core.tools import tool
    from langchain.agents import create_agent

    @tool
    def search_docs(query: str) -> str:
        "Recherche des passages dans la base documentaire interne."
        return "\n".join(d for _, d in retriever.search(query, k=2))

    agent = create_agent(make_lc_chat(), tools=[search_docs])
    out = agent.invoke({"messages": [("user",
        "D'après la doc interne, quelle est la différence entre LangGraph et CrewAI ?")]})
    print(lc_text(out["messages"][-1]))

try:
    demo_agentic_rag()
except ImportError as e:
    print("⏭️  LangGraph non installé :", e)
except Exception as e:
    print("⚠️  Besoin d'une clé d'API valide :", type(e).__name__, e)

D'après la documentation interne, voici la différence entre **LangGraph** et **CrewAI** :

---

### 🔷 LangGraph
LangGraph modélise un agent comme un **graphe d'états avec des arêtes conditionnelles**. Cela signifie que le flux d'exécution est représenté sous forme de nœuds (états) et de transitions conditionnelles entre ces nœuds. Cette approche offre un contrôle fin sur la logique de décision et les transitions d'un agent unique.

---

### 🟠 CrewAI
CrewAI, quant à lui, orchestre **plusieurs agents dotés de rôles, d'objectifs et de tâches**. L'accent est mis sur la collaboration multi-agents, où chaque agent a une identité et une mission propre, et où ils travaillent ensemble pour accomplir un objectif commun.

---

### 📊 En résumé

| Critère | LangGraph | CrewAI |
|---|---|---|
| **Paradigme** | Graphe d'états conditionnel | Orchestration multi-agents |
| **Unité centrale** | Un agent modélisé comme un graphe | Plusieurs agents avec rôles et objectifs |
| **Point fort** | Contrôle pré

### B.3 — Contrôle de ***groundedness***

Un risque du RAG : répondre au-delà des sources. On **vérifie** que la réponse est *ancrée*
dans le contexte récupéré. Version simple (recouvrement lexical) ; en production on utilise un
**LLM-juge** (cf. Lab 4).

In [8]:
def groundedness(answer, contexts):
    "Fraction des mots significatifs de la réponse présents dans le contexte (proxy simple)."
    ctx_words = set(w for c in contexts for w in _tok(c))
    ans_words = [w for w in _tok(answer) if len(w) > 3]
    if not ans_words:
        return 0.0
    covered = sum(w in ctx_words for w in ans_words)
    return round(covered / len(ans_words), 2)

ctx = [d for _, d in retriever.search("LangGraph", 2)]
print("Ancré      :", groundedness("LangGraph modélise un agent comme un graphe d'états.", ctx))
print("Halluciné  :", groundedness("LangGraph est un langage de programmation créé par Google.", ctx))
print("Seuil conseillé : alerter si < 0.5")

Ancré      : 1.0
Halluciné  : 0.2
Seuil conseillé : alerter si < 0.5


## C. CrewAI — système **multi-agents**

Trois agents collaborent : un **chercheur** rassemble, un **rédacteur** produit, un **relecteur**
vérifie. C'est le pattern *rôles + tâches* de CrewAI.

### C.1 — Processus **séquentiel**

In [9]:
async def demo_crew_sequential():
    from crewai import Agent, Task, Crew, Process
    m = crewai_model_string()

    chercheur = Agent(role="Chercheur", goal="Rassembler 3 faits clés sur le sujet",
                      backstory="Analyste rigoureux.", llm=m, verbose=False)
    redacteur = Agent(role="Rédacteur", goal="Rédiger un paragraphe clair et structuré",
                      backstory="Vulgarisateur technique.", llm=m, verbose=False)
    relecteur = Agent(role="Relecteur", goal="Corriger et garantir l'exactitude",
                      backstory="Éditeur exigeant.", llm=m, verbose=False)

    sujet = "les systèmes multi-agents fondés sur des LLM"
    t1 = Task(description=f"Liste 3 faits clés sur {sujet}.",
              agent=chercheur, expected_output="3 puces factuelles.")
    t2 = Task(description="Rédige un paragraphe à partir des faits.",
              agent=redacteur, expected_output="Un paragraphe de 4-6 phrases.", context=[t1])
    t3 = Task(description="Relis et corrige le paragraphe.",
              agent=relecteur, expected_output="Version finale corrigée.", context=[t2])

    crew = Crew(agents=[chercheur, redacteur, relecteur], tasks=[t1, t2, t3],
                process=Process.sequential, verbose=False)
    print(await crew.kickoff_async())

try:
    await demo_crew_sequential()
except ImportError as e:
    print("⏭️  CrewAI non installé :", e, "\n   pip install crewai")
except Exception as e:
    print("⚠️  Besoin d'une clé d'API valide :", type(e).__name__, e)

Après relecture attentive, le paragraphe est bien construit, cohérent et techniquement précis. Je n'y relève aucune erreur grammaticale, syntaxique ou factuelle. Voici néanmoins une version avec quelques légères améliorations stylistiques pour en renforcer la fluidité :

---

Les systèmes multi-agents fondés sur des grands modèles de langage (LLM) reposent sur la collaboration de plusieurs agents autonomes, chacun assigné à un rôle spécialisé — planificateur, exécuteur ou critique — afin de résoudre des tâches complexes qu'un agent unique ne pourrait traiter efficacement seul. Des frameworks comme AutoGen, CrewAI ou LangGraph permettent d'orchestrer ces interactions en définissant les flux de communication et les responsabilités de chaque agent. La coordination entre ces agents s'effectue principalement via le langage naturel ou semi-structuré, ce qui leur permet de partager leurs raisonnements, de déléguer des sous-tâches et de se corriger mutuellement de manière flexible et dynamique

### C.2 — Processus **hiérarchique** (un manager délègue)

En mode `hierarchical`, CrewAI ajoute un **manager** (LLM) qui **planifie**, **délègue** aux
agents et **agrège**. On ne définit plus l'ordre des tâches : le manager s'en charge.

In [10]:
async def demo_crew_hierarchical():
    from crewai import Agent, Task, Crew, Process
    m = crewai_model_string()

    chercheur = Agent(role="Chercheur", goal="Trouver des faits", backstory="Curieux.",
                      llm=m, verbose=False)
    redacteur = Agent(role="Rédacteur", goal="Rédiger clairement", backstory="Pédagogue.",
                      llm=m, verbose=False)

    tache = Task(description="Produire un court paragraphe fiable sur le pattern superviseur/worker.",
                 expected_output="Un paragraphe relu.")
    crew = Crew(agents=[chercheur, redacteur], tasks=[tache],
                process=Process.hierarchical, manager_llm=m, verbose=False)
    print(await crew.kickoff_async())

try:
    await demo_crew_hierarchical()
except ImportError as e:
    print("⏭️  CrewAI non installé :", e)
except Exception as e:
    print("⚠️  Besoin d'une clé d'API valide :", type(e).__name__, e)

Voici le paragraphe relu et finalisé sur le pattern superviseur/worker :

---

Le **pattern Superviseur/Worker** (aussi appelé Master/Worker) est un modèle de conception utilisé en programmation concurrente et dans les systèmes distribués. Son principe repose sur une séparation claire des rôles : le **superviseur** prend en charge une tâche complexe, la décompose en sous-tâches indépendantes, les distribue aux workers disponibles, surveille leur progression, collecte et agrège leurs résultats, et gère les pannes éventuelles en redistribuant les sous-tâches si un worker échoue. Les **workers**, quant à eux, reçoivent chacun une sous-tâche, l'exécutent de manière autonome sans connaissance des autres workers, puis retournent leur résultat au superviseur. Ce modèle est largement répandu : on le retrouve notamment dans Apache Hadoop avec le paradigme MapReduce, dans les serveurs web comme Nginx, ou encore dans les systèmes de files de tâches asynchrones tels que Celery et Sidekiq. Ses prin

### C.3 — Le **superviseur à la main** (pour démystifier)

CrewAI n'est « que » de l'orchestration de prompts. On reproduit le pattern
**superviseur → workers** en **pur Python** — et ça tourne **hors-ligne** (mock). C'est exactement
ce que font les frameworks, en plus habillé.

In [11]:
from llm_helpers import make_client

def worker(role, instruction, script=None):
    client = make_client(offline_script=script, quiet=True)
    return client.complete([
        {"role": "system", "content": f"Tu es {role}. Sois concis et factuel."},
        {"role": "user", "content": instruction},
    ]).content

def supervisor(sujet):
    print("👔 Superviseur : décompose la mission en 3 étapes.\n")
    faits = worker("Chercheur", f"Donne 3 faits clés sur {sujet}.",
                   script=["1) rôles 2) communication 3) orchestration par un superviseur"])
    print("🔎 Chercheur →", faits)
    draft = worker("Rédacteur", f"Rédige un paragraphe à partir de ces faits : {faits}",
                   script=["Les systèmes multi-agents répartissent le travail entre des rôles "
                           "spécialisés qui communiquent, sous la houlette d'un superviseur."])
    print("✍️  Rédacteur →", draft)
    final = worker("Relecteur", f"Corrige et valide ce paragraphe : {draft}",
                   script=["[VALIDÉ] " + "Les systèmes multi-agents répartissent le travail "
                           "entre des rôles spécialisés coordonnés par un superviseur."])
    print("✅ Relecteur →", final)
    return final

supervisor("les systèmes multi-agents fondés sur des LLM");

👔 Superviseur : décompose la mission en 3 étapes.

🔎 Chercheur → ## 3 faits clés sur les systèmes multi-agents fondés sur des LLM

1. **Décomposition des tâches complexes** : Les agents LLM peuvent se spécialiser et collaborer en parallèle (ex. : un agent planificateur, un agent exécuteur, un agent vérificateur), permettant de résoudre des problèmes qui dépassent la fenêtre de contexte ou les capacités d'un seul modèle.

2. **Émergence de comportements non programmés** : Des études (Park et al., *Generative Agents*, 2023 ; AutoGen, Microsoft 2023) montrent que des agents LLM en interaction peuvent exhiber des comportements sociaux complexes (négociation, délégation, correction mutuelle) sans que ces comportements soient explicitement codés.

3. **Vulnérabilité aux erreurs en cascade** : La propagation d'hallucinations entre agents constitue un risque majeur — une erreur générée par un agent peut être validée et amplifiée par les agents suivants, rendant la robustesse et la vérification

## D. Exercices

> Tentez la cellule `# TODO`, puis dépliez la solution.

**D.1 — RAG agentique custom.** Reprenez `build_custom_graph` (A.2) et remplacez l'outil
`multiply` par l'outil `search_docs` (B.2). Vous obtenez un **agent RAG** entièrement maîtrisé
(graphe explicite + récupération sous contrôle de l'agent).

In [12]:
def build_rag_graph():
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, START, END, MessagesState
    from langgraph.prebuilt import ToolNode

    @tool
    def search_docs(query: str) -> str:
        "Recherche dans la base documentaire interne."
        return "\n".join(d for _, d in retriever.search(query, k=2))

    tools = [search_docs]
    model = make_lc_chat().bind_tools(tools)

    def agent_node(state: MessagesState):
        return {"messages": [model.invoke(state["messages"])]}

    def should_continue(state: MessagesState):
        last = state["messages"][-1]
        return "tools" if getattr(last, "tool_calls", None) else END

    builder = StateGraph(MessagesState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()

try:
    rag_graph = build_rag_graph()
    result = rag_graph.invoke({"messages": [("user", "Que dit la doc sur CrewAI ?")]})
    print(lc_text(result["messages"][-1]))
except Exception as e:
    print("D.1 necessite LangGraph et une API LLM valide :", type(e).__name__, e)

Voici ce que la documentation indique sur **CrewAI** :

> 🤖 **CrewAI** est un framework qui **orchestre plusieurs agents** dotés de **rôles**, d'**objectifs** et de **tâches**.

En résumé, CrewAI permet de coordonner des agents IA en leur assignant des rôles et des responsabilités spécifiques, afin qu'ils collaborent pour accomplir des tâches complexes.

---

💡 À noter : la doc mentionne également **Craft AI**, qui est une plateforme **MLOps/LLMOps souveraine européenne** (déploiement, RAG, monitoring) — à ne pas confondre avec CrewAI !

Souhaitez-vous plus de détails sur l'un ou l'autre de ces outils ?


<details><summary>✅ Solution D.1</summary>

```python
def build_rag_graph():
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, START, END, MessagesState
    from langgraph.prebuilt import ToolNode

    @tool
    def search_docs(query: str) -> str:
        "Recherche dans la base documentaire interne."
        return "\n".join(d for _, d in retriever.search(query, k=2))

    tools = [search_docs]
    model = make_lc_chat().bind_tools(tools)
    def agent_node(s): return {"messages": [model.invoke(s["messages"])]}
    def cont(s): return "tools" if getattr(s["messages"][-1], "tool_calls", None) else END

    b = StateGraph(MessagesState)
    b.add_node("agent", agent_node); b.add_node("tools", ToolNode(tools))
    b.add_edge(START, "agent")
    b.add_conditional_edges("agent", cont, {"tools": "tools", END: END})
    b.add_edge("tools", "agent")
    return b.compile()

g = build_rag_graph()
print(g.invoke({"messages": [("user", "Que dit la doc sur CrewAI ?")]})["messages"][-1].content)
```
</details>

**D.2 — Comparaison.** Pour la même tâche (« explique le pattern superviseur/worker »),
mesurez **latence** et **nombre d'appels LLM** pour : (a) `react_loop` du Lab 2,
(b) le graphe LangGraph custom, (c) la crew CrewAI. Que paie-t-on pour le confort du framework ?

In [13]:
import inspect, time
from llm_helpers import make_client

QUESTION = "Explique le pattern superviseur/worker en 3 phrases."

async def bench(label, fn):
    t0 = time.perf_counter()
    try:
        result = fn()
        if inspect.isawaitable(result):
            result = await result
        text, calls = result
        status = "ok"
    except Exception as e:
        text, calls = f"{type(e).__name__}: {e}", "?"
        status = "erreur"
    dt = time.perf_counter() - t0
    print(f"{label:26} | {dt:6.2f}s | appels LLM: {calls} | {status}")
    print(str(text)[:500], "\n")

def run_from_scratch():
    client = make_client(
        offline_script=["Le superviseur decompose la mission, delegue aux workers, puis agrege les resultats."],
        quiet=True,
    )
    rep = client.complete([{"role": "user", "content": QUESTION}])
    return rep.content, client.n_calls

def run_langgraph_custom():
    graph = build_rag_graph() if "build_rag_graph" in globals() else build_custom_graph()
    out = graph.invoke({"messages": [("user", QUESTION)]})
    return lc_text(out["messages"][-1]), ">=1"

async def run_crewai_sequential():
    from crewai import Agent, Task, Crew, Process
    m = crewai_model_string()
    analyste = Agent(role="Analyste", goal="Expliquer le pattern superviseur/worker", backstory="Pedagogue.", llm=m, verbose=False)
    relecteur = Agent(role="Relecteur", goal="Verifier la clarte et l'exactitude", backstory="Editeur rigoureux.", llm=m, verbose=False)
    t1 = Task(description=QUESTION, agent=analyste, expected_output="Une explication en 3 phrases.")
    t2 = Task(description="Relis et corrige l'explication.", agent=relecteur, expected_output="Version finale concise.", context=[t1])
    crew = Crew(agents=[analyste, relecteur], tasks=[t1, t2], process=Process.sequential, verbose=False)
    return str(await crew.kickoff_async()), ">=2"

await bench("from scratch", run_from_scratch)
await bench("LangGraph custom", run_langgraph_custom)
await bench("CrewAI sequential", run_crewai_sequential)

from scratch               |   4.83s | appels LLM: 1 | ok
Le pattern **Superviseur/Worker** divise le travail entre un processus central (le superviseur) qui orchestre et distribue les tâches, et plusieurs processus subordonnés (les workers) qui les exécutent en parallèle. Le superviseur surveille l'état des workers, gère la file de tâches et peut relancer un worker en cas d'échec, assurant ainsi la résilience du système. Ce pattern est idéal pour paralléliser des traitements lourds et améliorer la tolérance aux pannes, comme dans les systèmes de trait 

LangGraph custom           |   6.49s | appels LLM: >=1 | ok
Voici le pattern **superviseur/worker** en 3 phrases :

1. **Division des rôles** : Un **superviseur** orchestre et coordonne le travail global, tandis que des **workers** spécialisés exécutent des tâches précises et bien définies qui leur sont assignées.

2. **Délégation intelligente** : Le superviseur décompose un objectif complexe en sous-tâches, les distribue aux workers l

<details><summary>💡 Indications D.2</summary>

- Latence : `import time; t0=time.time(); ...; print(time.time()-t0)`.
- Nombre d'appels : comptez les tours (Lab 2 : longueur de la trace ; CrewAI : 1 appel/tâche min,
  davantage en hiérarchique à cause du manager).
- Attendu : CrewAI hiérarchique = le plus d'appels (le manager planifie/agrège) → le plus coûteux
  mais le plus autonome ; le `react_loop` from-scratch = le plus léger mais le moins outillé.
</details>

## ✅ Récapitulatif

- **LangGraph** : un agent = **état + nœuds + arêtes conditionnelles**. Le *prebuilt* dépanne,
  mais le **graphe explicite** donne le contrôle (streaming, persistance, *human-in-the-loop*).
- **RAG** : récupérer puis **ancrer** la réponse ; le rendre **agentique** (outil) ; **vérifier**
  la *groundedness*.
- **Multi-agents** : rôles + tâches (CrewAI), séquentiel vs hiérarchique — au fond, de
  l'**orchestration de prompts** que vous savez reproduire à la main.

➡️ **Lab 4** : évaluer, tracer (coût/latence), **sécuriser** (injection, garde-fous,
*human-in-the-loop*) et **déployer** sur Craft AI.